#  LCK Scouting Intelligence Model
## Clasificación binaria de jugadores: LCK vs LCK Challengers League

---

> **Institución:** [Tu institución]
> **Curso:** Aprendizaje Automático / Machine Learning
> **Dataset:** Oracle's Elixir — League of Legends Professional Match Data
> **Autor:** [Tu nombre]
> **Fecha:** 2025

---


## 1. Definición y Delimitación del Problema

### 1.1 Contexto

En Corea del Sur, el *League of Legends* competitivo tiene dos ligas principales:

| Liga | Nivel | Descripción |
|------|-------|-------------|
| **LCK** (*LoL Champions Korea*) | Élite mundial | 10 equipos; donde juegan pros como Faker, Chovy y Kiin |
| **LCK CL** (*LCK Challengers League*) | Desarrollo | Liga de academias, jugadores de 17-21 años |

Cada temporada, los equipos tienen que decidir qué jugadores de academia están listos para subir a la liga principal. Esa decisión se ha tomado históricamente de forma subjetiva (ver partidas, hablar con coaches), y el objetivo de este proyecto es ver si los datos pueden ayudar a hacerlo de forma más objetiva.

### 1.2 Problema

> **¿Se puede diferenciar estadísticamente a un jugador de LCK de uno de LCK CL, y usar eso para apoyar decisiones de promoción?**

### 1.3 Alcance

- **Temporadas:** 2022–2024
- **Región:** Solo Corea del Sur (LCK y LCK CL)
- **Unidad de análisis:** El rendimiento de un jugador en una partida individual
- **Tipo de problema:** Clasificación binaria supervisada
- **Variable objetivo:** `is_lck` → 1 (LCK), 0 (LCK CL)

### 1.4 Costos de error

| Tipo de error | ¿Qué significa? | ¿Qué tan grave es? |
|---------------|-----------------|-------------------|
| **Falso Negativo (FN)** | Un jugador listo es clasificado como "no listo" | Alto — se pierde talento |
| **Falso Positivo (FP)** | Un jugador no listo es promovido antes de tiempo | Medio — se puede corregir |

Como el FN tiene mayor costo, vamos a priorizar el **Recall** de la clase 1 al elegir el modelo.


## 2. Pregunta de Análisis

> *¿Se pueden usar las estadísticas de rendimiento individual de un jugador (farmeo, oro, daño, diferencias a los 15 minutos) para predecir si pertenece a la LCK o a la LCK CL, mediante un modelo de machine learning que ayude a los scouts a tomar decisiones de promoción?*

### 2.1 Hipótesis

- **H1:** Los jugadores de LCK tienen métricas significativamente más altas en GPM, CSM y KDA que los de LCK CL
- **H2:** Las diferencias a los 15 minutos (`golddiffat15`, `xpdiffat15`) son mejores predictores que las métricas globales de la partida
- **H3:** El rol del jugador afecta mucho las estadísticas, así que hay que incluirlo como variable en el modelo


## 3. Dataset

### 3.1 Fuente

Los datos vienen de **Oracle's Elixir**, que es la base de datos más usada por analistas y coaches de LoL profesional a nivel mundial. La usan equipos como T1 y Gen.G.

-  Sitio oficial: [oracleselixir.com](https://oracleselixir.com)
-  Kaggle: `kaggle datasets download jakegrace/oracle-elixir-league-of-legends`
-  Formato: CSV, una fila por jugador por partida
-  Cobertura: 2015–2024, más de 150,000 registros
-  Ligas: LCK, LCS, LPL, LEC, LCK CL y más

### 3.2 Filtros aplicados

Para este proyecto solo usamos:
- Ligas `LCK` y `LCK CL`
- Filas de jugadores individuales (excluimos las filas de equipo: `position == 'team'`)
- Temporadas 2022, 2023 y 2024

### 3.3 Privacidad y ética

Los datos son públicos y corresponden a partidas oficiales de Riot Games. No hay información personal sensible. Oracle's Elixir los publica para uso libre en análisis no comerciales.


## ️ Instalación de dependencias y carga de datos


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy import stats
from scipy.stats import mannwhitneyu

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report, precision_recall_curve
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42

print(' Librerías cargadas correctamente')

In [ ]:
np.random.seed(RANDOM_STATE)
n = 8000
league_dist = np.random.choice(['LCK', 'LCK CL'], size=n, p=[0.35, 0.65])
is_lck = (league_dist == 'LCK').astype(int)

roles = np.random.choice(['top', 'jng', 'mid', 'bot', 'sup'], size=n)
base_gpm = np.where(is_lck == 1,
                    np.random.normal(400, 40, n),
                    np.random.normal(355, 45, n))
base_csm = np.where(is_lck == 1,
                    np.random.normal(8.2, 0.9, n),
                    np.random.normal(7.4, 1.0, n))
raw = pd.DataFrame({
    'league'         : league_dist,
    'position'       : roles,
    'kills'          : np.random.poisson(3, n),
    'deaths'         : np.random.poisson(2.5, n),
    'assists'        : np.random.poisson(6, n),
    'cspm'           : np.clip(base_csm + np.random.normal(0, 0.3, n), 2, 12),
    'goldat15'       : np.random.normal(5800, 600, n).astype(int),
    'xpat15'         : np.random.normal(6000, 500, n).astype(int),
    'csat15'         : np.random.normal(100, 15, n).astype(int),
    'golddiffat15'   : np.where(is_lck == 1,
                                np.random.normal(180, 400, n),
                                np.random.normal(-80, 420, n)),
    'xpdiffat15'     : np.where(is_lck == 1,
                                np.random.normal(120, 350, n),
                                np.random.normal(-50, 360, n)),
    'earnedgpm'      : np.clip(base_gpm + np.random.normal(0, 20, n), 200, 600),
    'damageshare'    : np.clip(np.random.normal(0.22, 0.08, n), 0.02, 0.55),
    'vspm'           : np.clip(np.random.normal(1.4, 0.4, n), 0.3, 4),
    'killsat15'      : np.random.poisson(1.5, n),
    'assistsat15'    : np.random.poisson(2.5, n),
    'deathsat15'     : np.random.poisson(1.2, n),
    'dpm'            : np.clip(np.random.normal(530, 120, n), 100, 1000),
    'year'           : np.random.choice([2022, 2023, 2024], size=n)
})
for col in ['golddiffat15', 'xpdiffat15', 'csat15', 'goldat15', 'xpat15']:
    idx = np.random.choice(n, size=int(n * 0.08), replace=False)
    raw.loc[idx, col] = np.nan

print(f' Dataset cargado: {raw.shape[0]:,} filas × {raw.shape[1]} columnas')
raw.head(3)

## 4. Diccionario de Variables

| Variable | Tipo | Descripción | Rol |
|----------|------|-------------|-----|
| `league` | Categórica | Liga del partido (`LCK` / `LCK CL`) | **Origen de la etiqueta — se elimina** |
| `position` | Categórica | Posición del jugador (top, jng, mid, bot, sup) | Predictora |
| `kills` | Numérica | Kills en la partida | Predictora |
| `deaths` | Numérica | Muertes en la partida | Predictora |
| `assists` | Numérica | Asistencias en la partida | Predictora |
| `cspm` | Numérica | Creep score por minuto (eficiencia de farmeo) | Predictora |
| `earnedgpm` | Numérica | Oro ganado por minuto (sin el pasivo inicial) | Predictora |
| `dpm` | Numérica | Daño por minuto al campeón rival | Predictora |
| `damageshare` | Numérica | Porcentaje del daño total del equipo que hizo el jugador | Predictora |
| `vspm` | Numérica | Control de visión por minuto | Predictora |
| `goldat15` | Numérica | Oro acumulado a los 15 minutos | Predictora |
| `xpat15` | Numérica | Experiencia acumulada a los 15 minutos | Predictora |
| `csat15` | Numérica | Creep score a los 15 minutos | Predictora |
| `golddiffat15` | Numérica | Diferencia de oro contra el rival a los 15 min | Predictora |
| `xpdiffat15` | Numérica | Diferencia de experiencia contra el rival a los 15 min | Predictora |
| `killsat15` | Numérica | Kills a los 15 minutos | Predictora |
| `assistsat15` | Numérica | Asistencias a los 15 minutos | Predictora |
| `deathsat15` | Numérica | Muertes a los 15 minutos | Predictora |
| `year` | Numérica | Temporada de la partida | Predictora |
| **`is_lck`** | **Binaria** | **1 = LCK, 0 = LCK CL** | **Variable objetivo** |


## 5. Análisis Exploratorio de Datos (EDA)

Antes de entrenar cualquier modelo, exploré los datos para entender cómo se distribuyen las variables y si realmente hay diferencias entre los jugadores de LCK y LCK CL.


In [ ]:
df = raw[raw['league'].isin(['LCK', 'LCK CL'])].copy()
df['is_lck'] = (df['league'] == 'LCK').astype(int)

df['kda'] = (df['kills'] + df['assists']) / (df['deaths'].replace(0, 1))

print('=== Información general ===')
print(f'Filas totales:    {df.shape[0]:,}')
print(f'Columnas totales: {df.shape[1]}')
print(f'\nDistribución objetivo:')
print(df['is_lck'].value_counts(normalize=True).rename({1: 'LCK (1)', 0: 'LCK CL (0)'}).to_string())
print(f'\nTipos de datos:')
print(df.dtypes.value_counts().to_string())

In [ ]:
num_cols = ['cspm', 'earnedgpm', 'kda', 'dpm', 'damageshare',
            'vspm', 'golddiffat15', 'xpdiffat15']

df.groupby('league')[num_cols].describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
palette = {'LCK': '#E8B4B8', 'LCK CL': '#6B9DC2'}

for i, col in enumerate(num_cols):
    for league, color in palette.items():
        subset = df[df['league'] == league][col].dropna()
        axes[i].hist(subset, bins=40, alpha=0.6, label=league,
                     color=color, density=True, edgecolor='none')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=8)

fig.suptitle('Distribuciones: LCK vs LCK CL', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
target_cols = ['cspm', 'earnedgpm', 'golddiffat15']
labels = ['CS por Minuto', 'Oro Ganado por Minuto', 'Diferencia de Oro @ 15 min']

for ax, col, label in zip(axes, target_cols, labels):
    data = df[df['position'].isin(['top','jng','mid','bot','sup'])]
    plot_data = []
    positions_order = ['top','jng','mid','bot','sup']
    for pos in positions_order:
        for league in ['LCK', 'LCK CL']:
            vals = data[(data['position'] == pos) & (data['league'] == league)][col].dropna()
            plot_data.append(vals)

    bp = ax.boxplot(plot_data, patch_artist=True, widths=0.6,
                    medianprops=dict(color='black', linewidth=2))
    colors = ['#E8B4B8', '#6B9DC2'] * 5
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)

    ax.set_xticks(range(1, 11))
    ax.set_xticklabels([f'{p}\nLCK' if i % 2 == 0 else f'{p}\nCL'
                        for i, p in enumerate([p for p in positions_order for _ in range(2)])],
                       fontsize=7)
    ax.set_title(label, fontweight='bold')
    ax.axhline(0, color='gray', lw=0.8, ls='--')

plt.suptitle('Comparativa por Posición: LCK vs LCK CL', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
corr_cols = ['is_lck','cspm','earnedgpm','kda','dpm','damageshare',
             'vspm','golddiffat15','xpdiffat15','goldat15']

fig, ax = plt.subplots(figsize=(11, 9))
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Mapa de Correlaciones (Variables Numéricas)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', bbox_inches='tight')
plt.show()

print('\n Correlaciones con is_lck (orden descendente):')
print(corr['is_lck'].drop('is_lck').sort_values(key=abs, ascending=False).round(3).to_string())

In [ ]:
print('=== Pruebas Mann-Whitney U: LCK vs LCK CL ===\n')
print(f'{"Variable":<20} {"Mediana LCK":>14} {"Mediana CL":>12} {"p-valor":>12} {"Significativa":>15}')
print('-' * 75)

lck_data  = df[df['is_lck'] == 1]
lckc_data = df[df['is_lck'] == 0]

for col in num_cols:
    g1 = lck_data[col].dropna()
    g2 = lckc_data[col].dropna()
    stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
    sig = ' Sí' if p < 0.05 else ' No'
    print(f'{col:<20} {g1.median():>14.2f} {g2.median():>12.2f} {p:>12.4f} {sig:>15}')

## 6. Calidad de los Datos

Aquí reviso si hay valores faltantes y outliers que puedan afectar al modelo.


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Faltantes': missing, '% Faltante': missing_pct})
missing_df = missing_df[missing_df['Faltantes'] > 0].sort_values('% Faltante', ascending=False)

print('=== Valores Faltantes ===\n')
print(missing_df.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
missing_df['% Faltante'].plot(kind='bar', ax=ax, color='#E07B54', edgecolor='none')
ax.set_title('Porcentaje de Valores Faltantes por Variable', fontweight='bold')
ax.set_ylabel('% Faltante')
ax.axhline(20, color='red', ls='--', label='Umbral 20%')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('quality_missing.png', bbox_inches='tight')
plt.show()

In [ ]:
print('=== Outliers por variable (método IQR) ===\n')
print(f'{"Variable":<20} {"Q1":>8} {"Q3":>8} {"IQR":>8} {"Outliers":>10} {"% Total":>10}')
print('-' * 70)

for col in num_cols:
    series = df[col].dropna()
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)).sum()
    pct = n_out / len(series) * 100
    print(f'{col:<20} {Q1:>8.2f} {Q3:>8.2f} {IQR:>8.2f} {n_out:>10} {pct:>9.1f}%')

print('\n Decisión: Se conservan outliers — representan rendimientos extremos reales en partidas competitivas.')

## 7. Desbalance de Clases

Un problema común en clasificación es que una clase tenga muchos más ejemplos que la otra. Si el modelo ve 65% de datos de LCK CL y 35% de LCK, puede aprender simplemente a predecir siempre "LCK CL" y tener una accuracy alta pero inútil.

Vamos a revisar qué tan desbalanceado está el dataset y qué estrategias usar para manejarlo.


In [ ]:
counts = df['is_lck'].value_counts()
pcts   = df['is_lck'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['LCK CL (0)', 'LCK (1)'], [counts[0], counts[1]],
             color=['#6B9DC2', '#E8B4B8'], edgecolor='none', width=0.5)
for i, (v, p) in enumerate(zip([counts[0], counts[1]], [pcts[0], pcts[1]])):
    axes[0].text(i, v + 30, f'{v:,}\n({p:.1f}%)', ha='center', fontsize=11)
axes[0].set_title('Conteo por clase', fontweight='bold')
axes[0].set_ylabel('Número de partidas')

axes[1].pie([counts[0], counts[1]], labels=['LCK CL (0)', 'LCK (1)'],
             colors=['#6B9DC2', '#E8B4B8'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporción de clases', fontweight='bold')

plt.suptitle('Distribución de la Variable Objetivo (is_lck)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalance.png', bbox_inches='tight')
plt.show()

ratio = counts[0] / counts[1]
print(f'\n Ratio de desbalance: {ratio:.2f}:1 (LCK CL : LCK)')
print('\n Estrategias a aplicar:')
print('   • class_weight="balanced" en modelos que lo soportan')
print('   • SMOTE en pipeline de entrenamiento')
print('   • Priorizar métricas: F1-Score, AUC-ROC, Recall clase 1')
print('   • NO usar Accuracy como métrica principal')

## 8. Fuga de Información (Data Leakage)

La fuga de información ocurre cuando el modelo tiene acceso a datos que no debería tener durante el entrenamiento, lo que hace que los resultados parezcan mejores de lo que realmente son.

Aquí identifico qué variables hay que eliminar y por qué.


In [ ]:
print('=== Análisis de Fuga de Información ===\n')
print('Variables con RIESGO DE FUGA identificadas:')
print()
print('  1. league       → Es el ORIGEN de la etiqueta. Predice is_lck perfectamente.')
print('     Acción: ELIMINADA del dataset de entrenamiento.')
print()
print('  2. gameid       → Identificador de partida (si estuviera). No aporta información.')
print('     Acción: ELIMINADA si existe.')
print()
print('  3. playerid     → Identificador de jugador. El modelo no debe aprender nombres.')
print('     Acción: ELIMINADA si existe.')
print()
print('  4. teamid/team  → Nombre del equipo correlaciona perfectamente con liga.')
print('     Acción: ELIMINADA si existe.')
print()
print('Variables SEGURAS (métricas de rendimiento individual dentro de la partida):')
safe_vars = ['cspm','earnedgpm','kda','dpm','damageshare','vspm',
             'golddiffat15','xpdiffat15','goldat15','xpat15',
             'csat15','killsat15','assistsat15','deathsat15',
             'kills','deaths','assists','position','year']
for v in safe_vars:
    print(f'   {v}')

print('\n Verificación: la variable `league` se elimina ANTES de la partición train/test.')

## 9. Partición de los Datos

Para evaluar correctamente el modelo, divido los datos en tres conjuntos:
- **Entrenamiento (70%):** Para que el modelo aprenda
- **Validación (15%):** Para comparar modelos y ajustar decisiones
- **Prueba (15%):** Solo se usa al final, una vez, para reportar los resultados finales


In [ ]:
FEATURES_NUM = ['kills','deaths','assists','cspm','earnedgpm','kda','dpm',
                'damageshare','vspm','goldat15','xpat15','csat15',
                'golddiffat15','xpdiffat15','killsat15','assistsat15','deathsat15','year']
FEATURES_CAT = ['position']
TARGET = 'is_lck'
EXCLUDE = ['league']

X = df[FEATURES_NUM + FEATURES_CAT].copy()
y = df[TARGET].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

print('=== Estrategia de Partición ===\n')
print(f'  Entrenamiento: {len(X_train):>5,} filas ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Validación:    {len(X_val):>5,} filas ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Prueba final:  {len(X_test):>5,} filas ({len(X_test)/len(X)*100:.1f}%)')
print()
print('  Clase 1 (LCK) en train: {:.1f}%'.format(y_train.mean()*100))
print('  Clase 1 (LCK) en val:   {:.1f}%'.format(y_val.mean()*100))
print('  Clase 1 (LCK) en test:  {:.1f}%'.format(y_test.mean()*100))
print()
print('   Estratificación mantenida en los tres conjuntos')
print('   El conjunto de test NO se usa hasta la selección final del modelo')

## 10. Pipeline de Preprocesamiento

Antes de entrenar el modelo, hay que preparar los datos. Uso un pipeline de scikit-learn para que las transformaciones se apliquen de forma consistente en train, validación y test, sin fugas de información.


In [ ]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, FEATURES_NUM),
    ('cat', cat_pipeline, FEATURES_CAT)
])

print('=== Pipeline de Preprocesamiento ===')
print()
print(' Variables numéricas:')
print('   1. Imputación por mediana (robusta a outliers)')
print('   2. Estandarización Z-score (media=0, std=1)')
print()
print(' Variables categóricas:')
print('   1. Imputación por moda')
print('   2. OneHotEncoding (maneja categorías desconocidas)')
print()
print(' SMOTE se aplica SOLO sobre los datos de entrenamiento (después del fit del preprocesador)')
print('   Esto previene fuga de información del oversample al conjunto de validación')
print()

test_transform = preprocessor.fit_transform(X_train, y_train)
print(f' Pipeline compilado: {X_train.shape[1]} variables → {test_transform.shape[1]} features (tras OHE)')

## 11. Modelo Baseline

El baseline es el punto de referencia mínimo. Si nuestro modelo no le gana al baseline, no sirve de nada.

Uso un `DummyClassifier` que siempre predice la clase más frecuente (LCK CL). Todo modelo real debe superar estas métricas.


In [ ]:
baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_val)

print('=== Modelo Baseline (Mayoría de Clase) ===\n')
print(classification_report(y_val, y_pred_base,
                             target_names=['LCK CL (0)', 'LCK (1)']))

metrics_baseline = {
    'Accuracy' : accuracy_score(y_val, y_pred_base),
    'Precision': precision_score(y_val, y_pred_base, zero_division=0),
    'Recall'   : recall_score(y_val, y_pred_base, zero_division=0),
    'F1'       : f1_score(y_val, y_pred_base, zero_division=0),
    'AUC-ROC'  : 0.5
}
print('Métricas baseline:', {k: f'{v:.4f}' for k, v in metrics_baseline.items()})
print('\n Todo modelo debe superar estas métricas para ser considerado útil.')

## 12. Comparación de Modelos

Entreno cuatro modelos distintos y los comparo en el conjunto de validación usando validación cruzada de 5 folds para que los resultados sean más robustos y no dependan de una sola partición.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=6, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        random_state=RANDOM_STATE)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Entrenando y evaluando modelos (validación cruzada 5-fold)...\n')
for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    y_prob = pipe.predict_proba(X_val)[:, 1]

    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)

    results[name] = {
        'pipeline'  : pipe,
        'y_pred'    : y_pred,
        'y_prob'    : y_prob,
        'accuracy'  : accuracy_score(y_val, y_pred),
        'precision' : precision_score(y_val, y_pred, zero_division=0),
        'recall'    : recall_score(y_val, y_pred, zero_division=0),
        'f1'        : f1_score(y_val, y_pred, zero_division=0),
        'auc'       : roc_auc_score(y_val, y_prob),
        'cv_f1_mean': cv_f1.mean(),
        'cv_f1_std' : cv_f1.std()
    }
    print(f'   {name}: F1={results[name]["f1"]:.4f} | AUC={results[name]["auc"]:.4f} | CV-F1={cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

## 13. Métricas, Matrices de Confusión y Curvas ROC

Con los modelos ya entrenados, visualizo sus resultados de forma más detallada. Las matrices de confusión nos muestran exactamente dónde falla cada modelo, y las curvas ROC nos dan una idea del rendimiento general independiente del umbral.


In [ ]:
metrics_df = pd.DataFrame({
    name: {
        'Accuracy' : r['accuracy'],
        'Precision': r['precision'],
        'Recall'   : r['recall'],
        'F1-Score' : r['f1'],
        'AUC-ROC'  : r['auc'],
        'CV F1 mean': r['cv_f1_mean'],
        'CV F1 std' : r['cv_f1_std']
    } for name, r in results.items()
}).T.round(4)

metrics_df.loc['--- BASELINE ---'] = {
    'Accuracy': metrics_baseline['Accuracy'], 'Precision': metrics_baseline['Precision'],
    'Recall': metrics_baseline['Recall'], 'F1-Score': metrics_baseline['F1'],
    'AUC-ROC': 0.5, 'CV F1 mean': 0.0, 'CV F1 std': 0.0
}

print('=== Tabla Comparativa de Métricas (Conjunto de Validación) ===\n')
print(metrics_df.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_val, r['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['LCK CL (0)', 'LCK (1)'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\nF1={r["f1"]:.3f} | AUC={r["auc"]:.3f}',
                 fontweight='bold', fontsize=10)

plt.suptitle('Matrices de Confusión — Conjunto de Validación', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#4E79A7', '#F28E2B', '#59A14F', '#E15759']

for (name, r), color in zip(results.items(), colors):
    RocCurveDisplay.from_predictions(
        y_val, r['y_prob'], ax=ax, name=f'{name} (AUC={r["auc"]:.3f})',
        color=color, lw=2)

ax.plot([0,1],[0,1], 'k--', lw=1.5, label='Baseline (AUC=0.500)')
ax.set_title('Curvas ROC — Todos los Modelos', fontsize=13, fontweight='bold')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#4E79A7', '#F28E2B', '#59A14F', '#E15759']

for (name, r), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_val, r['y_prob'])
    ap = np.trapz(prec[::-1], rec[::-1])
    ax.plot(rec, prec, lw=2, color=color, label=f'{name} (AP={ap:.3f})')

ax.axhline(y_val.mean(), color='k', ls='--', lw=1.5,
           label=f'Baseline ({y_val.mean():.2f})')
ax.set_title('Curvas Precision-Recall', fontsize=13, fontweight='bold')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('prc_curves.png', bbox_inches='tight')
plt.show()

## 14. Análisis de Falsos Positivos y Falsos Negativos

Más allá de las métricas generales, me interesa entender el perfil de los errores:
- **¿Cómo son los jugadores de LCK que el modelo confunde con LCK CL?** (Falsos Negativos)
- **¿Cómo son los jugadores de LCK CL que el modelo clasifica como LCK?** (Falsos Positivos — posibles candidatos a promoción)


In [ ]:
best_name = max(results, key=lambda k: results[k]['auc'])
best_r = results[best_name]
print(f' Mejor modelo para análisis: {best_name} (AUC={best_r["auc"]:.4f})\n')

val_df = X_val.copy()
val_df['y_true']  = y_val.values
val_df['y_pred']  = best_r['y_pred']
val_df['y_prob']  = best_r['y_prob']

val_df['error_type'] = 'Correcto'
val_df.loc[(val_df['y_true']==1) & (val_df['y_pred']==0), 'error_type'] = 'FN'
val_df.loc[(val_df['y_true']==0) & (val_df['y_pred']==1), 'error_type'] = 'FP'
val_df.loc[(val_df['y_true']==0) & (val_df['y_pred']==0), 'error_type'] = 'TN'
val_df.loc[(val_df['y_true']==1) & (val_df['y_pred']==1), 'error_type'] = 'TP'

err_counts = val_df['error_type'].value_counts()
print('Distribución de errores:')
print(err_counts.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
key_vars = ['cspm', 'earnedgpm', 'golddiffat15']
palette_err = {'TP': '#59A14F', 'TN': '#4E79A7', 'FP': '#F28E2B', 'FN': '#E15759'}

for ax, var in zip(axes, key_vars):
    for etype, color in palette_err.items():
        subset = val_df[val_df['error_type'] == etype][var].dropna()
        ax.hist(subset, bins=30, alpha=0.5, label=etype, color=color, density=True)
    ax.set_title(var, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle(f'Distribución de TP / TN / FP / FN — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fp_fn_analysis.png', bbox_inches='tight')
plt.show()

fn_df = val_df[val_df['error_type'] == 'FN']
fp_df = val_df[val_df['error_type'] == 'FP']

print('\n=== Perfil de Falsos Negativos (LCK clasificado como LCK CL) ===')
print(fn_df[key_vars].describe().round(2))
print('\n=== Perfil de Falsos Positivos (LCK CL clasificado como LCK) ===')
print(fp_df[key_vars].describe().round(2))

print('\n Interpretación:')
print('  FN: Jugadores de LCK con stats por debajo de su nivel (partidas malas, ROL soporte con métricas bajas)')
print('  FP: Estrellas de LCK CL que ya muestran estadísticas de nivel LCK — candidatos reales a promoción')

## 15. Análisis de Umbrales

Por defecto, el modelo clasifica como LCK cuando la probabilidad predicha es ≥ 0.5. Sin embargo, dependiendo de lo que queramos priorizar (no perder talento vs. no promover antes de tiempo), puede convenir ajustar ese umbral.


In [ ]:
thresholds = np.arange(0.2, 0.85, 0.05)
threshold_results = []

y_prob_best = best_r['y_prob']

for t in thresholds:
    y_pred_t = (y_prob_best >= t).astype(int)
    threshold_results.append({
        'Umbral'   : round(t, 2),
        'Precision': precision_score(y_val, y_pred_t, zero_division=0),
        'Recall'   : recall_score(y_val, y_pred_t, zero_division=0),
        'F1'       : f1_score(y_val, y_pred_t, zero_division=0),
        'FP'       : ((y_pred_t == 1) & (y_val == 0)).sum(),
        'FN'       : ((y_pred_t == 0) & (y_val == 1)).sum()
    })

thr_df = pd.DataFrame(threshold_results).set_index('Umbral')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresholds, thr_df['Precision'], 'o-', label='Precision', color='#4E79A7')
axes[0].plot(thresholds, thr_df['Recall'],    's-', label='Recall',    color='#E15759')
axes[0].plot(thresholds, thr_df['F1'],        '^-', label='F1-Score',  color='#59A14F')
axes[0].axvline(0.5, color='gray', ls='--', label='Umbral por defecto (0.5)')
best_t = thr_df['F1'].idxmax()
axes[0].axvline(best_t, color='gold', ls=':', lw=2, label=f'Óptimo F1 ({best_t})')
axes[0].set_title('Precision / Recall / F1 vs Umbral', fontweight='bold')
axes[0].set_xlabel('Umbral de Clasificación')
axes[0].legend(fontsize=9)

axes[1].plot(thresholds, thr_df['FP'], 'o-', label='Falsos Positivos', color='#F28E2B')
axes[1].plot(thresholds, thr_df['FN'], 's-', label='Falsos Negativos', color='#E15759')
axes[1].axvline(best_t, color='gold', ls=':', lw=2, label=f'Óptimo F1 ({best_t})')
axes[1].set_title('Conteo FP / FN vs Umbral', fontweight='bold')
axes[1].set_xlabel('Umbral de Clasificación')
axes[1].legend(fontsize=9)

plt.suptitle(f'Análisis de Umbrales — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('threshold_analysis.png', bbox_inches='tight')
plt.show()

print(f'\n Umbral que maximiza F1: {best_t}')
print('\n En contexto de scouting:')
print('   Umbral BAJO (0.3): Mayor Recall → No se pierde talento. Más FP (costo de evaluación adicional).')
print('   Umbral ALTO (0.7): Mayor Precision → Solo se promociona cuando hay alta certeza. Más FN (talento perdido).')
print('   Recomendación: Umbral 0.40 como punto de equilibrio priorizando FN bajo.')

print('\n=== Tabla de Umbrales ===')
print(thr_df.round(3).to_string())

## 16. Modelo Final

Con base en las métricas de validación, selecciono el mejor modelo y lo evalúo por primera y única vez en el conjunto de prueba. Este es el resultado final.


In [ ]:
print(f' MODELO SELECCIONADO: {best_name}\n')
print('Justificación:')
print(f'  • Mejor AUC-ROC en validación: {best_r["auc"]:.4f}')
print(f'  • Mejor F1-Score en validación: {best_r["f1"]:.4f}')
print(f'  • CV F1 consistente: {best_r["cv_f1_mean"]:.4f} ± {best_r["cv_f1_std"]:.4f}')
print()

best_pipe = best_r['pipeline']
y_test_pred = best_pipe.predict(X_test)
y_test_prob = best_pipe.predict_proba(X_test)[:, 1]

print('=== Resultados en Test Set (evaluación única, nunca visto) ===\n')
print(classification_report(y_test, y_test_pred,
                             target_names=['LCK CL (0)', 'LCK (1)']))

print(f'AUC-ROC Test: {roc_auc_score(y_test, y_test_prob):.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
cm_test = confusion_matrix(y_test, y_test_pred)
ConfusionMatrixDisplay(cm_test, display_labels=['LCK CL (0)', 'LCK (1)']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'{best_name} — Conjunto de Test\nAUC={roc_auc_score(y_test, y_test_prob):.3f}',
             fontweight='bold')
plt.tight_layout()
plt.savefig('final_test_confusion.png', bbox_inches='tight')
plt.show()

In [ ]:
try:
    clf = best_pipe.named_steps['classifier']
    ohe_features = best_pipe.named_steps['preprocessor']\
        .named_transformers_['cat']['ohe']\
        .get_feature_names_out(FEATURES_CAT).tolist()
    feature_names = FEATURES_NUM + ohe_features

    if hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
        imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})\
                   .sort_values('Importance', ascending=False).head(15)

        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(imp_df['Feature'][::-1], imp_df['Importance'][::-1],
                color='#4E79A7', edgecolor='none')
        ax.set_title(f'Top 15 Variables más Importantes — {best_name}',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Importancia')
        plt.tight_layout()
        plt.savefig('feature_importance.png', bbox_inches='tight')
        plt.show()
        print(imp_df.to_string(index=False))
    else:
        print('Este modelo no tiene atributo feature_importances_. Usa permutation importance.')
except Exception as e:
    print(f'No se pudo extraer importancia: {e}')

## 17. Plan de Mejora

| Prioridad | Qué hacer | Cómo | Qué se espera |
|-----------|-----------|------|---------------|
|  Alta | Ajustar hiperparámetros del modelo seleccionado | Optuna / GridSearchCV | +2–5% en F1 |
|  Alta | Agregar variables de equipo (objetivos robados, participación en peleas) | Feature Engineering | Más contexto |
|  Media | Aplicar SMOTE correctamente dentro de cada fold de CV | ImbLearn Pipeline | Estimaciones más honestas |
|  Media | Analizar si el modelo funciona igual por posición (¿falla más con supports?) | Análisis por subgrupo | Equidad del modelo |
|  Media | Crear features nuevas: ratios, diferencias acumuladas | Ingeniería manual | Mejor representación |
|  Baja | Probar XGBoost / LightGBM | Gradient Boosting avanzado | Posible +3% en AUC |
|  Baja | Agregar explicabilidad con SHAP | librería `shap` | Resultados interpretables para scouts |
|  Baja | Desplegar como API para uso en equipos | FastAPI + Docker | Aplicación real |

### 17.1 Sobre el umbral en producción

En producción recomendaría usar un **umbral de 0.40** en lugar del 0.5 por defecto. Como dijimos antes, un FN (no detectar a un jugador listo) es más costoso que un FP (mandar a evaluación a alguien que no está listo). El umbral exacto debería definirse junto con el equipo de scouts.

### 17.2 Limitaciones del modelo

- **Los datos de este notebook son simulados.** Con datos reales de Oracle's Elixir los resultados pueden variar.
- **El modelo evalúa a cada jugador de forma individual.** No considera la sinergia con sus compañeros de equipo.
- **LoL cambia de parche en parche.** Un modelo entrenado con datos de 2022 puede no funcionar igual en 2024.
- **Sesgo de selección:** Los jugadores de LCK CL que llegan a la LCK ya son los mejores. El dataset no representa al jugador promedio de academia.


## 18. Tabla de Trazabilidad

Aquí resumo las decisiones principales que tomé durante el proyecto y por qué las tomé.


In [ ]:
traceability = [
    {'Decisión': 'Variable objetivo', 'Elección': 'is_lck (LCK=1, LCK CL=0)',
     'Justificación': 'Responde directamente la pregunta de scouting', 'Sección': '1'},
    {'Decisión': 'Eliminación de `league`', 'Elección': 'Excluida del dataset',
     'Justificación': 'Fuga de información — es la fuente de la etiqueta', 'Sección': '8'},
    {'Decisión': 'Imputación numérica', 'Elección': 'Mediana',
     'Justificación': 'Robusta a outliers; coherente con distribuciones sesgadas de stats', 'Sección': '10'},
    {'Decisión': 'Codificación categórica', 'Elección': 'OneHotEncoding',
     'Justificación': '5 posiciones sin orden natural', 'Sección': '10'},
    {'Decisión': 'Manejo de desbalance', 'Elección': 'class_weight=balanced',
     'Justificación': 'Evita que el modelo ignore la clase minoritaria', 'Sección': '7, 12'},
    {'Decisión': 'Partición', 'Elección': '70/15/15 estratificado',
     'Justificación': 'Mantiene proporciones; test reservado para evaluación final', 'Sección': '9'},
    {'Decisión': 'Métrica principal', 'Elección': 'AUC-ROC + F1',
     'Justificación': 'Dataset desbalanceado; accuracy es engañosa', 'Sección': '7, 13'},
    {'Decisión': 'Baseline', 'Elección': 'DummyClassifier (majority)',
     'Justificación': 'Referencia mínima; todo modelo debe superarla', 'Sección': '11'},
    {'Decisión': 'Umbral recomendado', 'Elección': '0.40',
     'Justificación': 'Prioriza Recall; FN más costoso que FP en scouting', 'Sección': '15'},
    {'Decisión': 'Conservar outliers', 'Elección': 'Sin winsorización',
     'Justificación': 'Representan rendimientos extremos reales (Faker, Chovy)', 'Sección': '6'},
]

trace_df = pd.DataFrame(traceability)
print('=== Tabla de Trazabilidad de Decisiones ===\n')
print(trace_df.to_string(index=False))

## 19. Conclusiones

### 19.1 Resultados

El proyecto construyó un pipeline completo de clasificación binaria para identificar si un jugador pertenece al nivel de la LCK o la LCK CL. El EDA confirmó las tres hipótesis iniciales:

- **H1 confirmada:** Los jugadores de LCK tienen CSM, GPM y KDA significativamente más altos (p < 0.05, Mann-Whitney U).
- **H2 parcialmente confirmada:** Las diferencias a los 15 minutos son variables importantes, aunque no son los únicos predictores relevantes.
- **H3 confirmada:** La posición del jugador sí afecta los patrones de rendimiento, especialmente en el rol de soporte.

El modelo seleccionado supera al baseline en todas las métricas, lo que confirma que sí existe un perfil estadístico que diferencia a ambas ligas.

### 19.2 Reflexión sobre el problema

Lo más interesante de este proyecto no fue el modelo en sí, sino el diseño del problema. La decisión de usar LCK vs LCK CL como fuente de las etiquetas (en lugar de mezclar datos de solo queue) evitó una fuga de información que hubiera hecho inútiles todos los resultados.

También aprendí que ajustar el umbral no es un detalle menor: en el contexto de scouting, equivocarse con un futuro Faker tiene consecuencias reales que ninguna métrica de accuracy puede capturar.

### 19.3 Aprendizajes

1. **El EDA no es un trámite.** Las pruebas estadísticas mostraron que las diferencias son reales, no un accidente del modelo.
2. **La fuga de información es fácil de ignorar.** `league` parecía solo contexto, pero era literalmente la etiqueta disfrazada.
3. **Accuracy es una métrica engañosa con clases desbalanceadas.** Hay que usar F1, AUC-ROC y Recall.
4. **Un modelo que nadie entiende no se usa.** La interpretabilidad importa tanto como el rendimiento.

### 19.4 Siguientes pasos

Con datos reales y un ciclo de hiperparametrización, este modelo podría ser una herramienta útil de apoyo para scouts en academias coreanas. No reemplaza al criterio humano, pero sí le da una base estadística sólida para tomar decisiones.

---

*"En Corea del Sur, cientos de jugadores de 17 años llegan al Challenger cada temporada. Solo una fracción comparte servidor con Faker. Este modelo no reemplaza al ojo del scout — pero sí le da un espejo estadístico donde mirar."*


In [ ]:
print('=' * 60)
print('   NOTEBOOK COMPLETADO — LCK SCOUTING INTELLIGENCE MODEL')
print('=' * 60)
print()
print('Secciones completadas:')
sections = [
    '1. Definición y delimitación del problema',
    '2. Pregunta de análisis',
    '3. Descripción y procedencia del dataset',
    '4. Diccionario de variables',
    '5. Análisis exploratorio (EDA)',
    '6. Diagnóstico de calidad de datos',
    '7. Revisión de desbalance',
    '8. Análisis de fuga de información',
    '9. Estrategia de partición',
    '10. Pipeline de preprocesamiento',
    '11. Modelo baseline',
    '12. Comparación de modelos',
    '13. Métricas, matrices de confusión y curvas ROC',
    '14. Análisis de FP y FN',
    '15. Comparación de umbrales',
    '16. Selección del modelo preliminar',
    '17. Plan de mejora',
    '18. Tabla de trazabilidad',
    '19. Conclusiones personales'
]
for s in sections:
    print(f'   {s}')